In [4]:
import pickle
import sys
sys.path.append('../../scripts')
from gpa import gpa
import vtk_helper
import numpy as np
from typing import Dict,List
from tqdm.auto import tqdm
from sklearn.covariance import EmpiricalCovariance, EllipticEnvelope, MinCovDet
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.io import savemat
import vtk

In [13]:
mouse_wild_types = '/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/gut_wd'

mesh_file =   "/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/reference_gut/oriented_mesh.obj"

#outputs
perc_out =   "/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/wt_mah_percentiles.pkl"
perc_out_male =   "/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/reference_gut/percentiles.mat"
perc_out_male_face =   "/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/reference_gut/face_percentiles.mat" # important output for now!
perc_out_female = "/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/wt_female_percentiles.mat" # 
perc_out_female_face =   "/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/wt_female_f_percentiles.mat" # important output for now! 

In [3]:

o_f =   "/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/reference_gut/ref.pkl"
of = open(o_f,'rb')
male_female = pickle.load(of)
of.close()

In [4]:
# mahalanobises

_,_,res_males = gpa(male_female, 100, 10e-11)

    
    
    

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/19 [00:00<?, ?it/s]

Procrustes distance  1902073.0834770878


  0%|          | 0/19 [00:00<?, ?it/s]

Procrustes distance  4792.872885098832


...

  0%|          | 0/19 [00:00<?, ?it/s]

Procrustes distance  4.682500642006826e-10


  0%|          | 0/19 [00:00<?, ?it/s]

Procrustes distance  1.1706251002164705e-12


  0%|          | 0/20 [00:00<?, ?it/s]

In [5]:
def mahalanobis_distibutions(cohort: Dict[str,np.ndarray]) -> List[np.ndarray]:
    # build distributions
    num_of_points = list(cohort.values())[0].shape[0]
    dists = []
    for i in tqdm(range(num_of_points)):  # i in range of num_of_points
        points = []
        #print (i, working_dict)

        for mesh_v in cohort.values():
            points.append(mesh_v[i])

        points = np.vstack(points)
        #print(points)
        emp_cov = EmpiricalCovariance().fit(points)
        
        mah_dists = []
        for point in range(points.shape[0]):
            dst = emp_cov.mahalanobis([points[point]])
            mah_dists.append(dst)
        threshold = np.percentile(mah_dists, 99)
        
        #print(mah_dists)
        #mah_dists = np.array(mah_dists)[np.where(np.array(mah_dists) < 10)[0]]
        dists.append(np.array(mah_dists))
        #break
        
    return dists

res_mahalanobis_males =  mahalanobis_distibutions(res_males)


  0%|          | 0/6849 [00:00<?, ?it/s]

In [6]:
import seaborn as sns
import matplotlib.pyplot as plt

# Replace 'data' with your actual data

# Flatten the nested list
flat_data = [item for sublist in res_mahalanobis for np.array(item) in sublist]

# Create the histogram
sns.histplot(np.reshape(flat_data, (-1)), bins=20)  # You can adjust the number of bins as needed
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.title('Histogram of Data')
plt.show()

SyntaxError: cannot assign to function call (<ipython-input-6-c6a754794b83>, line 7)

In [12]:

def get_percentiles_faces(percentiles : Dict[int,List[float]], obj_filename) -> Dict[int,List[float]]:
    
    mesh = vtk_helper.read_vtk_obj(obj_filename)
    
    faces = vtk_helper.get_mesh_faces(mesh)
    
    face_percentiles = dict(zip(percentiles.keys(),[[] for x in percentiles.keys()])) 
        

    for face in tqdm( faces):
        res_dict_tmp =  dict(zip(percentiles.keys(),[0.0 for x in percentiles.keys()])) 
        
        for vert_id in face:
            
            # compute sum for all vertices in faces
            for key in percentiles.keys():
                res_dict_tmp[key] = res_dict_tmp[key] + percentiles[key][vert_id]
            
        # compute average and sum
        for key in percentiles.keys():
            face_percentiles[key].append (  res_dict_tmp[key] /3 )
    return face_percentiles


In [8]:
def get_percentiles(mah_distancess : List[np.ndarray] ) -> Dict[int,List[float]]:
    #computes percentiles for points
    ###99 percentiles 
    percentiles = {75:[], 95:[], 99:[]}

    for mahalanobis in tqdm(mah_distancess):
        for perc in percentiles.keys():
            percentiles[perc].append(np.percentile(mahalanobis, perc,interpolation='higher'))
    return percentiles



perc_m = get_percentiles(res_mahalanobis_males)



  0%|          | 0/6849 [00:00<?, ?it/s]

In [11]:
savemat(perc_out_male,{ "perc" + str(k): v for k,v in perc_m.items()})

In [14]:
###compute average percentiles for faces!!
perc_out_male_f = get_percentiles_faces(perc_m,obj_filename=mesh_file)

savemat(perc_out_male_face,{ "perc" + str(k): v for k,v in perc_out_male_f.items()})



  0%|          | 0/15105 [00:00<?, ?it/s]

In [ ]:
sns.histplot(np.stack(mahalanobis), bins=20)  # You can adjust the number of bins as needed
plt.show()

In [ ]:
np.percentile(mahalanobis,99)

In [ ]:
indices = np.where(mahalanobis < np.percentile(mahalanobis,99,memo))[0]

In [ ]:
indices

In [ ]:
np.array(mahalanobis)[indices]

In [ ]:
perc_m[75]

In [ ]:
dict(zip(perc_m.keys(),[[],[],[]]))

In [ ]:
import importlib
importlib.reload(vtk_helper)

In [ ]:
np.percentile()

In [ ]:
np.__version__